# 02 - Run Methods

Run after `01_preprocessing.ipynb`.

1. imports one method from `src/s3_methods/` (per `implementation_plan.md`'s Methods table),
2. scores it on one patient-level CV fold via `src/s5_eval/run_experiment.py`.

Set `FOLD` below and repeat across all 5 folds; results are aggregated as mean +/- CI in
`03_results_analysis.ipynb`.

In [ ]:
import os, pathlib, sys

root = pathlib.Path.cwd()
while not (root / "requirements.txt").exists() and root != root.parent:
    root = root.parent
os.chdir(root)
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from src.paths import resolve_roots
DATA_ROOT, OUTPUT_ROOT = resolve_roots()

print('Repo root  :', root)
print('DATA_ROOT  :', DATA_ROOT)
print('OUTPUT_ROOT:', OUTPUT_ROOT)

In [ ]:
import itertools

from src.s1_data.dataset import load_fold_datasets
from src.s1_data.splits import load_folds
from src.s5_eval.run_experiment import run_experiment

# Swap this import for the method under test, e.g.:
# from src.s3_methods.m1_traditional.a_thresholding import segment
# from src.s3_methods.m5_deep_learning.d_unet import UNet

PROCESSED_DIRS = (
    DATA_ROOT / 'processed/duke_dme_denoised',
    DATA_ROOT / 'processed/hc_ms_denoised',
)

FOLD = 0  # which of the 5 folds (written by 01_preprocessing.ipynb) to score on
folds = load_folds(DATA_ROOT / 'processed/folds.json')
_, val_ids = folds[FOLD]

# val_ids spans both datasets. load_fold_datasets routes each id to the
# directory holding it and raises if any id is in neither (stale fold) or in
# both (same patient scored twice); chain() concatenates them into one pass.
test_set = itertools.chain(*load_fold_datasets(PROCESSED_DIRS, val_ids))

In [ ]:
# Results are written under OUTPUT_ROOT (defined in the setup notebook's Drive
# cell -> MyDrive/Segmentation/output), so they persist across Colab restarts and
# can be pulled to your machine with scripts/pull_from_drive.py.
# per_sample, summary = run_experiment(
#     method_name='intensity_thresholding',
#     segment_fn=segment,
#     dataset=test_set,
#     output_csv=OUTPUT_ROOT / 'intensity_thresholding.csv',
# )
# summary